# Canvas API Tasks

This notebook provides a simplified interface to common Canvas LMS tasks using the `canvastask` module.

## Overview

The `canvastask` module provides functions for:
- Testing API connection and listing courses
- Managing assignments and submissions
- Synchronizing markdown content with Canvas pages

## Setup Requirements

1. Canvas API token stored in `.env` file (one level up: `../.env`)
2. Format: `CANVAS_TOKEN=your_token_here`
3. Set `COURSE_ID` below to your target course ID

## Table of Contents

- [Setup Requirements](#setup-requirements) -- must run first
- [Test Connection](#test-connection)
- [List Active Courses](#list-active-courses)
- [List Assignments](#list-assignments)
- [Get Submissions](#get-submissions)
- [Download File Submissions](#download-file-submissions)
- [List Quizzes](#list-quizzes)
- [Download Quiz Responses](#download-quiz-responses)
- [Download Canvas Pages as Markdown](#download-canvas-pages-as-markdown)
- [Upload Markdown Files to Canvas](#upload-markdown-files-to-canvas)

In [1]:
# Check and install required packages
import subprocess
import sys
import requests
import markdown
import html2text
import frontmatter
from datetime import datetime


# Set course ID (e.g., `https://canvas.instructure.com/courses/14011875`)
COURSE_ID = 14011875 # for ECO 331 S26

In [2]:
# Import canvastask module and other dependencies
import canvastask
import os
from pathlib import Path

# Load Canvas API token from .env file
canvastask.load_env()

# Verify token was loaded
CANVAS_TOKEN = os.environ.get('CANVAS_TOKEN')
if CANVAS_TOKEN and CANVAS_TOKEN != 'your_token_here':
    print('✓ Canvas token loaded successfully')
else:
    print('⚠ Please add your Canvas token to the .env file')

Loaded .env file
✓ Canvas token loaded successfully


## Test Connection

Verify that the API connection works by fetching your user profile.

In [3]:
# Get current user info
user = canvastask.canvas_request('users/self')
if user:
    print(f"✓ Connected as: {user['name']}")
    print(f"  User ID: {user['id']}")
else:
    print("✗ Failed to connect to Canvas")

✓ Connected as: Jonathan Conning
  User ID: 15131563


## List Active Courses

View all your active Canvas courses to find the course ID for ECO 331.

In [4]:
# List all active courses
courses = canvastask.canvas_request('courses', params={'enrollment_state': 'active'})
if courses:
    print('Your active courses:\n')
    for course in courses:
        course_id = course.get('id')
        name = course.get('name', 'Unnamed')
        code = course.get('course_code', '')
        print(f"  ID: {course_id}")
        print(f"  Name: {name}")
        print(f"  Code: {code}")
        print()
else:
    print("No courses found")

Your active courses:

  ID: 11204793
  Name: Eco 331: Economic History, SP25
  Code: Eco 331

  ID: 14011875
  Name: Eco 331: Economic History, Spring 2026
  Code: Eco 331:

  ID: 1345003
  Name: Your Guided Course Template
  Code: CANVAS-NA



## List Assignments

View all assignments for the selected course. Make sure you set `COURSE_ID` above first.

In [4]:
if COURSE_ID:
    assignments = canvastask.canvas_request(f'courses/{COURSE_ID}/assignments')
    if assignments:
        print(f'Published Assignments for course {COURSE_ID}:\n')
        
        # Filter for assignments where the 'published' attribute is True
        published_assignments = [a for a in assignments if a.get('published', False)]
        
        for a in published_assignments:
            print(f"  ID: {a['id']}")
            print(f"  Name: {a['name']}")
            print(f"  Due: {a.get('due_at', 'No due date')}")
            print()
            
        if not published_assignments:
            print("No published assignments found")
    else:
        print("No assignments found")
else:
    print('⚠ Please set COURSE_ID above')


Published Assignments for course 14011875:

  ID: 61516327
  Name: Introduce Yourself
  Due: 2026-02-03T15:00:00Z

  ID: 61516331
  Name: (Thurs 1/29) Intro: Q & Comments
  Due: 2026-01-29T15:00:00Z

  ID: 61516333
  Name: (Tue 2/3)  Neolithic Revolution: Q & Comments
  Due: 2026-02-03T15:30:00Z

  ID: 61516351
  Name: (Tue 2/24) Geography and Institutions: Q & Comments
  Due: 2025-02-24T15:00:00Z

  ID: 61516335
  Name: (Tue 3/10): Demographic shocks and institutional responses
  Due: 2026-03-10T14:00:00Z



## Get Submissions

Fetch student submissions for a specific assignment and save to a markdown file.

- Handles pagination to get all submissions (not just first 10)
- Saves to `canvas_submits/` folder
- Includes submission text, scores, and comments
- Set the `ASSIGNMENT_ID` below.

In [6]:
# Set assignment ID here (from the list above)
ASSIGNMENT_ID = 61516335  # Replace with an assignment ID

if COURSE_ID and ASSIGNMENT_ID:
    # Download all submissions for the assignment
    # Automatically detects submission type and extracts content appropriately
    filepath = canvastask.download_assignment_submissions(COURSE_ID, ASSIGNMENT_ID)
    if filepath:
        print(f"\nMarkdown file saved to: {filepath}")
else:
    print('⚠ Please set COURSE_ID and ASSIGNMENT_ID above')

Fetching submissions for: (Tue 3/10): Demographic shocks and institutional responses
  Fetched page 1 (54 submissions)

✓ Found 54 total submissions

✓ Saved 33 submissions to: c:\Users\jonat\My Drive\Hunter\eco331\code\canvas_submits\submissions_61516335_20260310_090811.md
  (21 students did not submit)

Markdown file saved to: canvas_submits\submissions_61516335_20260310_090811.md


## Download File Submissions

Download all file attachments from student submissions for an assignment.

This is useful for:
- Batch downloading all submissions for offline grading
- Processing submissions with external tools
- Creating backups of student work

In [7]:
import requests

# Set the assignment ID that has file uploads
DOWNLOAD_ASSIGNMENT_ID = 61516333  # Replace with assignment ID

if COURSE_ID and DOWNLOAD_ASSIGNMENT_ID:
    download_folder = Path(f'./submissions_{DOWNLOAD_ASSIGNMENT_ID}')
    download_folder.mkdir(exist_ok=True)
    
    # Get submissions with user info
    submissions = canvastask.canvas_request(
        f'courses/{COURSE_ID}/assignments/{DOWNLOAD_ASSIGNMENT_ID}/submissions',
        params={'include[]': ['user']}
    )
    
    if submissions:
        file_count = 0
        for sub in submissions:
            student_name = sub.get('user', {}).get('name', 'Unknown')
            # Clean student name for use in filename
            safe_name = "".join(c for c in student_name if c.isalnum() or c in (' ', '-', '_')).strip()
            
            attachments = sub.get('attachments', [])
            if not attachments:
                continue
                
            for attachment in attachments:
                file_url = attachment['url']
                filename = attachment['filename']
                
                # Save with student name prefix
                save_as = download_folder / f"{safe_name}_{filename}"
                
                print(f"Downloading: {save_as.name}")
                response = requests.get(
                    file_url,
                    headers={'Authorization': f"Bearer {os.environ.get('CANVAS_TOKEN')}"}
                )
                
                if response.status_code == 200:
                    save_as.write_bytes(response.content)
                    file_count += 1
                else:
                    print(f"  Error downloading: {response.status_code}")
        
        print(f"\nDownloaded {file_count} files to {download_folder.absolute()}")
    else:
        print("No submissions found")
else:
    print('⚠ Please set COURSE_ID and DOWNLOAD_ASSIGNMENT_ID above')


Downloaded 0 files to h:\My Drive\Hunter\eco331\code\submissions_61516333


## List Quizzes

View all quizzes for the selected course to find the quiz ID.

In [8]:
if COURSE_ID:
    quizzes = canvastask.canvas_request(f'courses/{COURSE_ID}/quizzes')
    if quizzes:
        print(f'Quizzes for course {COURSE_ID}:\n')
        for q in quizzes:
            print(f"  ID: {q['id']}")
            print(f"  Title: {q['title']}")
            print(f"  Questions: {q.get('question_count', 0)}")
            print(f"  Points: {q.get('points_possible', 0)}")
            print(f"  Published: {q.get('published', False)}")
            print()
    else:
        print("No quizzes found")
else:
    print('⚠ Please set COURSE_ID above')

Quizzes for course 14011875:

  ID: 24683814
  Title: 2/28 Engerman & Sokoloff reading Quiz
  Questions: 5
  Points: 23.0
  Published: False

  ID: 24863156
  Title: Paper Presentation Ranking
  Questions: 2
  Points: 4.0
  Published: True

  ID: 24683812
  Title: Paper Presentation sign up
  Questions: 3
  Points: None
  Published: True



## Download Quiz Responses

Download all student responses from a Canvas quiz including questions and answers.

This is useful for:
- Reviewing student quiz responses offline
- Analyzing answer patterns across the class
- Grading essay or short-answer questions
- Creating records of student work

Handles different question types:
- Multiple choice, true/false
- Essay and short answer
- Matching, fill-in-blank
- Numerical/calculated questions
- Multiple answers (checkboxes)

In [9]:
# Set quiz ID here (get from Canvas quiz URL)
# Example URL: https://canvas.instructure.com/courses/14011875/quizzes/12345
QUIZ_ID = 24863156  # Replace with quiz ID

if COURSE_ID and QUIZ_ID:
    # Download all quiz responses with questions and answers
    filepath = canvastask.download_quiz_responses(COURSE_ID, QUIZ_ID)
    if filepath:
        print(f"\nMarkdown file saved to: {filepath}")
else:
    print('⚠ Please set COURSE_ID and QUIZ_ID above')

Fetching responses for quiz: Paper Presentation Ranking
  Fetched page 1 (46 submissions)

✓ Found 46 total submissions

  Fetching answers for 46 submissions...
  ✓ Fetched answers for 44 completed submissions

✓ Saved 44 responses to: h:\My Drive\Hunter\eco331\code\canvas_quizzes\quiz_responses_24863156_20260210_083932.md
  (2 students did not complete the quiz)

Markdown file saved to: canvas_quizzes\quiz_responses_24863156_20260210_083932.md


## Download Canvas Pages as Markdown

Download all Canvas pages from your course and save them as markdown files with YAML frontmatter.

This allows you to:
- Edit course content locally in your text editor or Obsidian
- Version control your course materials with git
- Work offline and sync changes back to Canvas

Files are saved to a `canvas_pages/` folder with frontmatter preserving publication status.

In [7]:
if COURSE_ID:
    canvastask.download_canvas_pages_to_markdown(COURSE_ID, output_dir='canvas_pages')
else:
    print('⚠ Please set COURSE_ID above')


  ✓ Course Outline And Reading Schedule
     → course_outline_and_reading_schedule.md (published)
  ✓ Course Slides
     → course_slides.md (draft)
  ✓ Course Syllabus
     → course_syllabus.md (published)
  ✓ Final Exam Questions
     → final_exam_questions.md (draft)
  ✓ Google Scholar And Zotero
     → google_scholar_and_zotero.md (published)
  ✓ Instructor And Contact Info
     → instructor_and_contact_info.md (draft)
  ✓ Midterm Materials
     → midterm_materials.md (published)
  ✓ Paper Presentations
     → paper_presentations.md (draft)
  ✓ Week 4 Malthusian Economics
     → week_4_malthusian_economics.md (draft)
  ✓ Paper Presentation Schedule
     → paper_presentation_schedule.md (draft)

Downloaded 10 pages to C:\Users\jonat\My Drive\Hunter\eco331\code\canvas_pages


## Upload Markdown Files to Canvas

Upload markdown files from your `canvas_pages/` folder back to Canvas as pages.

### YAML Frontmatter Format

Your markdown files can include optional YAML frontmatter to control:

```markdown
---
published: true
title: Custom Page Title
---

# Page content here
```

- `published`: Set to `true` to publish immediately, `false` to save as draft (optional)
- `title`: Custom title for the page (optional, defaults to filename)

If no frontmatter is specified, the page title is derived from the filename.

In [5]:
if COURSE_ID:
    # Upload all markdown files from canvas_pages/ folder to Canvas
    canvastask.upload_all_markdown_files(COURSE_ID, canvas_folder='canvas_pages')
else:
    print('⚠ Please set COURSE_ID above')

Uploading 11 markdown files to Canvas

Page 'Course Outline And Reading Schedule' uploaded successfully (published)
URL: https://canvas.instructure.com/courses/14011875/pages/course-outline-and-reading-schedule
  ✓ published: course_outline_and_reading_schedule.md
Page 'Course Slides' uploaded successfully (draft)
URL: https://canvas.instructure.com/courses/14011875/pages/course-slides
  ✓ draft: course_slides.md
Page 'Course Syllabus' uploaded successfully (published)
URL: https://canvas.instructure.com/courses/14011875/pages/course-syllabus
  ✓ published: course_syllabus.md
Page 'Final Exam Questions' uploaded successfully (draft)
URL: https://canvas.instructure.com/courses/14011875/pages/final-exam-questions
  ✓ draft: final_exam_questions.md
Page 'Google Scholar And Zotero' uploaded successfully (draft)
URL: https://canvas.instructure.com/courses/14011875/pages/google-scholar-and-zotero
  ✓ draft: google_scholar_and_zotero.md
Page 'Instructor And Contact Info' uploaded successfully

## Download Class List

Download a list of active students in the course to use as a template for your grading spreadsheet.

In [8]:
if COURSE_ID:
    # This will save a CSV with 'user_id', 'name', 'sortable_name', and 'email'
    # You can open this in Excel, add 'score' and 'comment' columns, and use it for grading.
    students = canvastask.download_students(COURSE_ID, output_csv='canvas_submits/class_list.csv')
    if students:
        import pandas as pd
        display(pd.DataFrame(students[:5])) # Show first 5 students
else:
    print('⚠ Please set COURSE_ID above')

Fetching students for course: 14011875
  Fetched page 1 (53 students)

✓ Found 53 total active students

✓ Saved student list to: c:\Users\jconning\My Drive\Hunter\eco331\code\canvas_submits\class_list.csv


: 

## Upload Grades from File

Upload grades and comments directly from a local Excel or CSV file to Canvas.

1. First, export your class list using the block above (or use the Gradebook export from Canvas).
2. Make sure your file has a column for the student's Canvas `user_id` and a column for the `score`.
3. (Optional) Add a column for `comment` if you want to leave text feedback.
4. Fill in the file path and column names below, then run the block.

In [ ]:
# 1. Set the Assignment ID you are grading
UPLOAD_ASSIGNMENT_ID = 61516333 # Replace with your assignment ID

# 2. Set the path to your graded Excel or CSV file
GRADES_FILE = 'canvas_submits/class_list.csv'

# 3. Set your column names (how they appear in your spreadsheet)
USER_ID_COLUMN = 'user_id'
SCORE_COLUMN = 'score'
COMMENT_COLUMN = 'comment' # Set to None if you don't have comments

if COURSE_ID and UPLOAD_ASSIGNMENT_ID:
    import pandas as pd
    import os
    from pathlib import Path
    
    file_path = Path(GRADES_FILE)
    if file_path.exists():
        # Preview the file first
        print("Preview of file:")
        df = pd.read_csv(file_path) if file_path.suffix == '.csv' else pd.read_excel(file_path)
        display(df.head())
        print("\nUploading...")
        
        # Perform the upload
        result = canvastask.upload_grades_from_file(
            COURSE_ID, 
            UPLOAD_ASSIGNMENT_ID, 
            GRADES_FILE, 
            user_id_col=USER_ID_COLUMN, 
            score_col=SCORE_COLUMN, 
            comment_col=COMMENT_COLUMN
        )
    else:
        print(f"⚠ File not found: {GRADES_FILE}")
else:
    print('⚠ Please set COURSE_ID and UPLOAD_ASSIGNMENT_ID above')